# Data valuation & Shapley for data — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Credit assignment for training examples
Data Shapley values assign credit by averaging each row's marginal contribution across coalitions. These examples compute utilities, marginal gains, exact Shapley values, a harmful point, and an approximate estimate.

In [ ]:
# 1) Utility of a subset: 1-nearest-neighbor accuracy on two test points
train=np.array([0.,1.,3.]); y=np.array([0,0,1]); test=np.array([.2,2.8]); yt=np.array([0,1])
def util(S):
    if len(S)==0: return 0.0
    pred=nearest_predict(train[list(S)],y[list(S)],test); return (pred==yt).mean()
vals=[util(S) for S in [[],[0],[2],[0,2]]]
plt.figure(figsize=(6,3)); plt.bar(['empty','{0}','{2}','{0,2}'],vals); plt.title('subset utility')
assert vals==[0.0,0.5,0.5,1.0]

In [ ]:
# 2) Marginal contribution depends on the coalition already present
train=np.array([0.,1.,3.]); y=np.array([0,0,1]); test=np.array([.2,2.8]); yt=np.array([0,1])
def util(S):
    if len(S)==0: return 0.0
    return (nearest_predict(train[list(S)],y[list(S)],test)==yt).mean()
gains=[util([2])-util([]), util([0,2])-util([0]), util([1,2])-util([1])]
plt.figure(figsize=(6,3)); plt.bar(['empty','with 0','with 1'],gains); plt.title('marginal gains for point 2')
assert gains==[0.5,0.5,0.5]

In [ ]:
# 3) Exact Shapley averages weighted marginal gains over all coalitions
train=np.array([0.,1.,3.]); y=np.array([0,0,1]); test=np.array([.2,2.8]); yt=np.array([0,1])
def util(S):
    if len(S)==0: return 0.0
    return (nearest_predict(train[list(S)],y[list(S)],test)==yt).mean()
phi=[]
for i in range(3):
    val=0; others=[j for j in range(3) if j!=i]
    for r in range(3):
        for S in itertools.combinations(others,r):
            val+=math.factorial(r)*math.factorial(2-r)/math.factorial(3)*(util(list(S)+[i])-util(list(S)))
    phi.append(val)
plt.figure(figsize=(6,3)); plt.bar(['point 0','point 1','point 2'],phi); plt.title('exact data Shapley values')
assert np.allclose(phi,[0.25,0.25,0.5]) and abs(sum(phi)-1.0)<1e-12

In [ ]:
# 4) A mislabeled point can have negative value
train=np.array([0.,2.7,3.]); y=np.array([0,0,1]); test=np.array([.2,2.8]); yt=np.array([0,1])
def util(S):
    if len(S)==0: return 0.0
    return (nearest_predict(train[list(S)],y[list(S)],test)==yt).mean()
gain=util([0,1,2])-util([0,2])
plt.figure(figsize=(6,3)); plt.bar(['add mislabeled point 1'],[gain]); plt.axhline(0,c='k'); plt.title('bad rows can lower utility')
assert gain<0

In [ ]:
# 5) Permutation sampling approximates Shapley without all subsets
rng=np.random.default_rng(0); train=np.array([0.,1.,3.]); y=np.array([0,0,1]); test=np.array([.2,2.8]); yt=np.array([0,1])
def util(S):
    if len(S)==0: return 0.0
    return (nearest_predict(train[list(S)],y[list(S)],test)==yt).mean()
est=np.zeros(3)
for _ in range(120):
    perm=rng.permutation(3); S=[]; before=util(S)
    for i in perm:
        after=util(S+[i]); est[i]+=after-before; S.append(i); before=after
est/=120
plt.figure(figsize=(6,3)); plt.bar(['point 0','point 1','point 2'],est); plt.title('permutation estimate')
assert np.allclose(est.sum(),1.0) and est[2]>est[0]